In [4]:
import pandas as pd
from tabulate import tabulate
import textdistance

# A curated list of interesting pairs from your provided data
string_pairs = [
    ("SG_UF_NOT", "SG_UF"),
    ("ID_MUNICIP", "ID_MN_RESI"),
    ("NU_LOTE_V", "NU_LOTE_H"),
    ("DT_OBITO", "DT_DIAG"),
    ("ID_RG_RESI", "ID_REGIONA"),
    ("TERCEIRO", "TERCEIRIZA"),
    ("DT_NOTIFIC", "NU_ANO"),
    ("CS_ESCOL_N", "CS_RACA")
]

def calculate_all_similarities(s1, s2):
    """Calculates a wide range of similarity metrics for two strings."""
    results = {
        "String 1": s1,
        "String 2": s2,
    }

    # --- Edit Distance Family ---
    # These algorithms work on the strings directly and need no special setup.
    results["Levenshtein"] = textdistance.levenshtein.normalized_similarity(s1, s2)
    results["Damerau-Lev"] = textdistance.damerau_levenshtein.normalized_similarity(s1, s2)
    results["Jaro-Winkler"] = textdistance.jaro_winkler(s1, s2)
    try:
        results["Hamming"] = textdistance.hamming.normalized_similarity(s1, s2)
    except ValueError:
        results["Hamming"] = 0.0

    # --- N-Gram (Trigram, qval=3) Family ---
    # CORRECTED SECTION:
    # We instantiate the algorithm with qval=3 to tell it to use trigrams.
    # Then we call .similarity() directly on the strings.
    q = 3
    results["Jaccard (n=3)"] = textdistance.Jaccard(qval=q).normalized_similarity(s1, s2)
    results["Dice (n=3)"] = textdistance.Sorensen(qval=q).normalized_similarity(s1, s2)
    results["Overlap (n=3)"] = textdistance.Overlap(qval=q).normalized_similarity(s1, s2)
    results["Cosine (n=3)"] = textdistance.Cosine(qval=q).normalized_similarity(s1, s2)

    # --- Phonetic Family ---
    # These work directly on strings.
    results["MRA"] = textdistance.mra.similarity(s1, s2)
    nysiis1 = textdistance.nysiis(s1)
    nysiis2 = textdistance.nysiis(s2)
    results["NYSIIS (sim)"] = textdistance.levenshtein.normalized_similarity(nysiis1, nysiis2)

    return results

# Calculate all metrics for the selected pairs
all_results = [calculate_all_similarities(s1, s2) for s1, s2 in string_pairs]

# Format for display
df = pd.DataFrame(all_results)
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].round(4)

print(tabulate(df, headers='keys', tablefmt='grid'))

AttributeError: module 'textdistance' has no attribute 'nysiis'

In [1]:
"""
Comparação ampla de similaridade entre pares de variáveis DATASUS
(Suporta Levenshtein, Damerau-Lev., Jaro-Winkler, Hamming,
métricas de trigramas e dois algoritmos fonéticos: MRA e NYSIIS).

Requisitos:
    pip install textdistance tabulate pandas
    # Opcional para NYSIIS:
    pip install jellyfish     # lib leve, C-backed, com NYSIIS pronto
"""

from __future__ import annotations
import pandas as pd
from tabulate import tabulate
import textdistance

# --- Phonetic back-ends -----------------------------------------------------
from textdistance import MRA                     # nativo (Match Rating)
try:
    import jellyfish                             # para NYSIIS
    HAVE_NYSIIS = True
except ImportError:
    HAVE_NYSIIS = False

# ---------------------------------------------------------------------------
STRING_PAIRS = [
    ("SG_UF_NOT", "SG_UF"),
    ("ID_MUNICIP", "ID_MN_RESI"),
    ("NU_LOTE_V", "NU_LOTE_H"),
    ("DT_OBITO", "DT_DIAG"),
    ("ID_RG_RESI", "ID_REGIONA"),
    ("TERCEIRO", "TERCEIRIZA"),
    ("DT_NOTIFIC", "NU_ANO"),
    ("CS_ESCOL_N", "CS_RACA"),
]

# --- Métricas helpers -------------------------------------------------------
def trigram_metric(cls, s1, s2, q=3):
    return cls(qval=q).normalized_similarity(s1, s2)

def jaro_winkler_sim(s1, s2):
    return textdistance.JaroWinkler().normalized_similarity(s1, s2)

def levenshtein_sim(s1, s2):
    return textdistance.Levenshtein().normalized_similarity(s1, s2)

def damerau_lev_sim(s1, s2):
    return textdistance.DamerauLevenshtein().normalized_similarity(s1, s2)

def hamming_sim(s1, s2):
    if len(s1) != len(s2):
        return 0.0
    return textdistance.Hamming().normalized_similarity(s1, s2)

def mra_sim(s1, s2):
    # MRA.similarity devolve score 0-6; normalizar para 0-1
    raw = MRA().similarity(s1, s2)
    return raw / 6

def nysiis_sim(s1, s2):
    if not HAVE_NYSIIS:
        return None
    code1, code2 = jellyfish.nysiis(s1), jellyfish.nysiis(s2)
    return levenshtein_sim(code1, code2)


def compute_metrics(a: str, b: str) -> dict[str, float | str]:
    return {
        "String 1": a,
        "String 2": b,
        "Levenshtein": levenshtein_sim(a, b),
        "Damerau-Lev": damerau_lev_sim(a, b),
        "Jaro-Winkler": jaro_winkler_sim(a, b),
        "Hamming": hamming_sim(a, b),
        "Jaccard (3-gram)": trigram_metric(textdistance.Jaccard, a, b),
        "Dice (3-gram)": trigram_metric(textdistance.Sorensen, a, b),
        "Overlap (3-gram)": trigram_metric(textdistance.Overlap, a, b),
        "Cosine (3-gram)": trigram_metric(textdistance.Cosine, a, b),
        "MRA (phonetic)": mra_sim(a, b),
        "NYSIIS (phonetic)": nysiis_sim(a, b),
    }

# --- Execução ---------------------------------------------------------------
records = [compute_metrics(x, y) for x, y in STRING_PAIRS]
df = pd.DataFrame(records).round(4)
print(tabulate(df, headers="keys", tablefmt="grid"))


+----+------------+------------+---------------+---------------+----------------+-----------+--------------------+-----------------+--------------------+-------------------+------------------+---------------------+
|    | String 1   | String 2   |   Levenshtein |   Damerau-Lev |   Jaro-Winkler |   Hamming |   Jaccard (3-gram) |   Dice (3-gram) |   Overlap (3-gram) |   Cosine (3-gram) |   MRA (phonetic) |   NYSIIS (phonetic) |
+====+============+============+===============+===============+================+===========+====================+=================+====================+===================+==================+=====================+
|  0 | SG_UF_NOT  | SG_UF      |        0.5556 |        0.5556 |         0.9111 |    0      |             0.4286 |          0.6    |             1      |            0.6547 |           0.5    |              0.5556 |
+----+------------+------------+---------------+---------------+----------------+-----------+--------------------+-----------------+--------

In [1]:
import json
import os
import pandas as pd
import numpy as np
from gensim.models.fasttext import FastText
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# ==============================================================================
# PHASE 0: SETUP
# ==============================================================================
# This script assumes 'datasus_analysis.json' is in the same directory.
INPUT_JSON_PATH = "datasus_analysis.json"
CORPUS_PATH = "varnames.txt" # A temporary file for training
TOP_N_RESULTS = 50

# Check if the required file exists before starting.
if not os.path.exists(INPUT_JSON_PATH):
    print(f"❌ CRITICAL ERROR: '{INPUT_JSON_PATH}' not found.")
    print("Please place this script in the same directory as your JSON file.")
    exit()

# ==============================================================================
# PHASE 1: LOAD YOUR DATA AND CREATE MASTER VARIABLE PROFILES
# ==============================================================================
print("PHASE 1: Starting Data Loading from your 'datasus_analysis.json'...")
with open(INPUT_JSON_PATH, 'r') as f:
    analysis_data = json.load(f)

master_profiles = {}
for series_name, series in analysis_data.items():
    # This logic mirrors the R script's approach to handle potentially failed analyses
    if series.get("series_metadata", {}).get("status") == "Success":
        for var_instance in series.get("variable_analysis", []):
            var_name = var_instance["variable_name"]
            if var_name not in master_profiles:
                master_profiles[var_name] = {
                    "name": var_name,
                    "appears_in_series": set(),
                }
            master_profiles[var_name]["appears_in_series"].add(series_name)
                
all_vars = sorted(list(master_profiles.keys()))
n_vars = len(all_vars)
print(f"✅ PHASE 1: Complete. Found {n_vars} unique variables in your data.\n")

# ==============================================================================
# PHASE 2: CREATE CORPUS FROM YOUR DATA AND TRAIN FASTTEXT MODEL
# ==============================================================================
print("PHASE 2: Starting fastText Model Training on your variable names...")
# 1. Create the corpus file from your actual variable names
with open(CORPUS_PATH, 'w') as f:
    for var in all_vars:
        f.write(var + '\n')

# 2. Train the fastText model
print(" -> Training model...")
model = FastText(
    corpus_file=CORPUS_PATH,
    vector_size=64, # Dimensionality of the embedding vectors
    window=3,
    min_count=1,      # Include all variables, no matter how rare
    min_n=3,          # Minimum character n-gram length
    max_n=5,          # Maximum character n-gram length
    sg=1,             # Use the skip-gram model (generally better for this task)
    workers=os.cpu_count() - 1 # Use available cores for faster training
)
print("✅ PHASE 2: Complete. Model is trained on your data.\n")

# ==============================================================================
# PHASE 3: GENERATE EMBEDDINGS AND CALCULATE CROSS-DATASET SIMILARITY
# ==============================================================================
print("PHASE 3: Starting Embedding Generation & Similarity Calculation...")
# 1. Generate an embedding vector for every variable
embedding_matrix = np.zeros((n_vars, model.vector_size))
for i, var_name in enumerate(all_vars):
    embedding_matrix[i] = model.wv[var_name]

# 2. Compute the dense cosine similarity matrix
name_similarity_matrix = cosine_similarity(embedding_matrix)

# 3. Apply the "cross-dataset only" rule, as specified in your R script's logic
print(" -> Applying the cross-dataset rule to ensure relevance...")
mask = np.ones_like(name_similarity_matrix)
for i in tqdm(range(n_vars), desc="Building cross-dataset mask"):
    for j in range(n_vars):
        if i == j:
            mask[i, j] = 0
            continue
        
        var_i_sources = master_profiles[all_vars[i]]["appears_in_series"]
        var_j_sources = master_profiles[all_vars[j]]["appears_in_series"]
        if var_i_sources.intersection(var_j_sources):
            mask[i, j] = 0 # Zero out similarity for variables sharing a source dataset

# Apply the mask to get the final, relevant similarities
final_similarity_matrix = name_similarity_matrix * mask
print("✅ PHASE 3: Complete.\n")

# ==============================================================================
# PHASE 4: PRESENT THE RESULTS
# ==============================================================================
print("PHASE 4: Generating Top Similarity Pairs from fastText Embeddings...")
sim_df = pd.DataFrame(final_similarity_matrix, index=all_vars, columns=all_vars)

# Melt the DataFrame to get a list of pairs: (VarA, VarB, Similarity)
top_pairs = sim_df.stack().reset_index()
top_pairs.columns = ['Variable_A', 'Variable_B', 'Similarity']

# Remove duplicate pairs (e.g., keep (A,B) and drop (B,A))
top_pairs = top_pairs[top_pairs['Variable_A'] < top_pairs['Variable_B']]

# Sort by similarity and display the most promising results
top_pairs_sorted = top_pairs.sort_values(by='Similarity', ascending=False).head(TOP_N_RESULTS)

print("\n--- Top Cross-Dataset Variable Name Similarity Pairs (fastText Cosine Similarity) ---")
print(top_pairs_sorted.to_string(index=False))
print("✅ PHASE 4: Complete.\n")

# Clean up the temporary corpus file
os.remove(CORPUS_PATH)

PHASE 1: Starting Data Loading from your 'datasus_analysis.json'...
✅ PHASE 1: Complete. Found 3407 unique variables in your data.

PHASE 2: Starting fastText Model Training on your variable names...
 -> Training model...
✅ PHASE 2: Complete. Model is trained on your data.

PHASE 3: Starting Embedding Generation & Similarity Calculation...
 -> Applying the cross-dataset rule to ensure relevance...


Building cross-dataset mask: 100%|██████████| 3407/3407 [00:12<00:00, 267.91it/s]


✅ PHASE 3: Complete.

PHASE 4: Generating Top Similarity Pairs from fastText Embeddings...

--- Top Cross-Dataset Variable Name Similarity Pairs (fastText Cosine Similarity) ---
Variable_A Variable_B  Similarity
 CLI_OUTRO CLI_OUTROS    0.858166
 ANT_OUTRO ANT_OUTROS    0.849064
 CLI_LOCAL MCLI_LOCAL    0.842634
 DEF_AUDIT DEF_AUDITI    0.835519
  DT_ATEND  DT_ATENDE    0.824696
 CON_LOCAL CON_LOCALI    0.822471
 ASSINTOMA ASSINTOMAT    0.816853
 CON_LOCAL CON_LOCAL2    0.813014
    CARVAO   F_CARVAO    0.810585
   ESPLENO   ESPLENOM    0.809745
CS_SECRECA ID_SECRECA    0.808910
 ANT_DOSES ANT_DOSES_    0.808285
 CLI_NEURO CLI_NEUROL    0.805938
 ANT_OUTRO ANT_OUTRO_    0.805402
  TP_LOCAL TP_LOCALLE    0.803392
  LAB_DT_2 LAB_DT_L_2    0.802747
  CEFALEIA STCEFALEIA    0.801612
AT_SINTOMA DT_SINTOMA    0.799167
    UF_RES   UF_RESID    0.797721
  DT_ATEND DT_ATENDIM    0.792834
CLI_NEFRIT  CLI_NEFRO    0.792200
ANT_CONTAG ANT_CONTAT    0.792178
 DT_INICIO DT_INICIO_    0.788057
  ANT_

In [1]:
"""
Análise de Discordância entre Métricas de Similaridade de Strings (v3 - Corrigido e Robusto)

Este script:
1. Carrega um corpus real de nomes de variáveis do 'datasus_analysis.json'.
2. Treina um modelo fastText.
3. Compara métricas de similaridade, agora com uma "guarda" para prevenir ZeroDivisionError
   em variáveis com nomes curtos (< 3 caracteres).
4. Calcula a discordância e exibe os 50 pares mais ambíguos.
"""
import json
import os
import random
import pandas as pd
import numpy as np
from tabulate import tabulate
from tqdm import tqdm

import textdistance
from gensim.models.fasttext import FastText

# --- Configuração ---
INPUT_JSON_PATH = "datasus_analysis.json"
CORPUS_PATH = "varnames.txt"
SAMPLE_SIZE = 2000
TOP_N_DISAGREEMENT = 50

# --- Funções Auxiliares de Similaridade ---

def levenshtein_sim(s1, s2):
    return textdistance.Levenshtein().normalized_similarity(s1, s2)

def damerau_lev_sim(s1, s2):
    return textdistance.DamerauLevenshtein().normalized_similarity(s1, s2)

def jaro_winkler_sim(s1, s2):
    return textdistance.JaroWinkler().normalized_similarity(s1, s2)

def hamming_sim(s1, s2):
    if len(s1) != len(s2):
        return 0.0
    return textdistance.Hamming().normalized_similarity(s1, s2)

def trigram_metric(cls, s1, s2, q=3):
    # FIX: Adiciona uma "guarda" para prevenir o ZeroDivisionError.
    # Se qualquer uma das strings for muito curta para formar um trigrama (n=3),
    # a similaridade de n-gram é logicamente 0.
    if len(s1) < q or len(s2) < q:
        return 0.0
    return cls(qval=q).normalized_similarity(s1, s2)

def fasttext_sim(s1, s2, model):
    v1 = model.wv[s1]
    v2 = model.wv[s2]
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

def compute_all_metrics(a: str, b: str, ft_model) -> dict[str, float | str]:
    metrics = {
        "Levenshtein": levenshtein_sim(a, b),
        "Damerau-Lev": damerau_lev_sim(a, b),
        "Jaro-Winkler": jaro_winkler_sim(a, b),
        "Hamming": hamming_sim(a, b),
        "Jaccard (n=3)": trigram_metric(textdistance.Jaccard, a, b),
        "Dice (n=3)": trigram_metric(textdistance.Sorensen, a, b),
        "Overlap (n=3)": trigram_metric(textdistance.Overlap, a, b),
        "fastText": fasttext_sim(a, b, ft_model),
    }
    std_dev = np.std(list(metrics.values()))
    return {"Variable A": a, "Variable B": b, "Std. Dev.": std_dev, **metrics}

# --- Script Principal ---

def main():
    if not os.path.exists(INPUT_JSON_PATH):
        print(f"❌ ERRO CRÍTICO: '{INPUT_JSON_PATH}' não encontrado.")
        return

    # 1. Carregar dados
    print("1. Carregando e processando 'datasus_analysis.json'...")
    with open(INPUT_JSON_PATH, 'r') as f:
        analysis_data = json.load(f)
    all_vars = sorted(list(set(
        var_instance["variable_name"]
        for series in analysis_data.values()
        if series.get("series_metadata", {}).get("status") == "Success"
        for var_instance in series.get("variable_analysis", [])
    )))
    print(f"   -> Encontradas {len(all_vars)} variáveis únicas.")

    # 2. Treinar modelo fastText
    print("\n2. Treinando modelo fastText sobre o corpus de variáveis...")
    with open(CORPUS_PATH, 'w') as f:
        for var in all_vars:
            f.write(var + '\n')
    
    ft_model = FastText(vector_size=64, window=5, min_count=1, min_n=3, max_n=5, sg=1, workers=os.cpu_count() - 1)
    ft_model.build_vocab(corpus_file=CORPUS_PATH)
    ft_model.train(corpus_file=CORPUS_PATH, total_examples=ft_model.corpus_count, total_words=ft_model.corpus_total_words, epochs=ft_model.epochs)
    
    os.remove(CORPUS_PATH)
    print("   -> Modelo treinado.")

    # 3. Gerar amostra de pares
    print(f"\n3. Gerando uma amostra de {SAMPLE_SIZE} pares únicos de variáveis...")
    sampled_pairs = set()
    if len(all_vars) > 1:
        # Garantir que a amostra não seja maior que o número total de pares possíveis
        max_pairs = (len(all_vars) * (len(all_vars) - 1)) // 2
        actual_sample_size = min(SAMPLE_SIZE, max_pairs)
        if actual_sample_size < SAMPLE_SIZE:
             print(f"   -> Aviso: O tamanho da amostra foi reduzido para {actual_sample_size} (máximo de pares possíveis).")
        
        while len(sampled_pairs) < actual_sample_size:
            pair = tuple(sorted(random.sample(all_vars, 2)))
            sampled_pairs.add(pair)
    print("   -> Amostra gerada.")

    # 4. Calcular métricas
    print("\n4. Calculando métricas de similaridade para a amostra...")
    records = [compute_all_metrics(a, b, ft_model) for a, b in tqdm(sampled_pairs, desc="Analisando Pares")]

    # 5. Exibir resultados
    print("\n5. Processando e exibindo os resultados...")
    if not records:
        print("   -> Nenhuma análise foi realizada.")
        return
        
    df = pd.DataFrame(records)
    df_sorted = df.sort_values(by="Std. Dev.", ascending=False)
    top_disagreement = df_sorted.head(TOP_N_DISAGREEMENT)

    print("\n--- Top 50 Pares com Maior Discordância entre Métricas de Similaridade ---")
    print("Um 'Std. Dev.' alto indica que os algoritmos discordam fortemente sobre a similaridade do par.")
    
    display_df = top_disagreement.round(4)
    print(tabulate(display_df, headers="keys", tablefmt="grid", showindex=False))

if __name__ == "__main__":
    main()

1. Carregando e processando 'datasus_analysis.json'...
   -> Encontradas 3407 variáveis únicas.

2. Treinando modelo fastText sobre o corpus de variáveis...
   -> Modelo treinado.

3. Gerando uma amostra de 2000 pares únicos de variáveis...
   -> Amostra gerada.

4. Calculando métricas de similaridade para a amostra...


Analisando Pares: 100%|██████████| 2000/2000 [00:00<00:00, 2113.51it/s]



5. Processando e exibindo os resultados...

--- Top 50 Pares com Maior Discordância entre Métricas de Similaridade ---
Um 'Std. Dev.' alto indica que os algoritmos discordam fortemente sobre a similaridade do par.
+--------------+--------------+-------------+---------------+---------------+----------------+-----------+-----------------+--------------+-----------------+------------+
| Variable A   | Variable B   |   Std. Dev. |   Levenshtein |   Damerau-Lev |   Jaro-Winkler |   Hamming |   Jaccard (n=3) |   Dice (n=3) |   Overlap (n=3) |   fastText |
+==============+==============+=============+===============+===============+================+===========+=================+==============+=================+============+
| HEMO_R2      | TEMPO_TRAT   |      0.2956 |        0.5    |        0.5    |         0.7381 |    0      |          0      |       0      |          0      |    -0.0646 |
+--------------+--------------+-------------+---------------+---------------+----------------+-------